# Étape 4 — Génération DOCXTransforme `<Livre>_EDIT.txt` en `<Livre>.docx`.**Aucune IA, aucune clé API, aucun coût.** Uniquement `python-docx` et la convention typographique. Deux exécutions produisent le même document : régénérez autant que vous voulez après avoir ajusté un réglage.---**Ce notebook n'est qu'une interface.** Toute la logique vit dans le paquet`theatre_editor`. On y monte le Drive, on installe les dépendances, on surchargeéventuellement la configuration, puis on lance l'étape.**Cette étape est reprenable.** Si Colab coupe, relancez la celluled'exécution : le travail déjà validé ne sera pas refait, et vous ne repaierezaucun appel.

## 1. Dépendances et montage du Drive

In [ ]:
# Installation des dépendances du pipeline.!pip install -q -U openai pymupdf python-docxfrom google.colab import drivedrive.mount("/content/drive")

## 2. Récupération du codeLe dépôt [`elyeskaak/texte_troupe_theatre`](https://github.com/elyeskaak/texte_troupe_theatre)est **privé** : son clone exige un jeton d'accès personnel.**À faire une fois.**1. GitHub → *Settings* → *Developer settings* → *Personal access tokens* →   *Fine-grained tokens* → **Generate new token**   - *Repository access* : **uniquement** `texte_troupe_theatre`   - *Permissions* → *Repository permissions* → **Contents : Read-only**   Rien de plus. Un jeton limité à la lecture d'un seul dépôt ne peut rien   casser s'il fuite.2. Colab → panneau latéral **🔑 Secrets** → ajouter `GITHUB_TOKEN` →   activer « Accès au notebook ».Si vous préférez ne pas créer de jeton, utilisez l'option B : déposez le dossier`theatre_editor/` directement sur votre Drive.

In [ ]:
# --- Option A : clone du dépôt privé -----------------------------------DEPOT_COMPTE = "elyeskaak"DEPOT_NOM = "texte_troupe_theatre"DOSSIER_PROJET = f"/content/{DEPOT_NOM}"import osimport subprocessimport sysfrom google.colab import userdatatry:    jeton = userdata.get("GITHUB_TOKEN")except Exception:    jeton = Noneif not jeton:    raise RuntimeError(        "Secret GITHUB_TOKEN introuvable.\n"        "Panneau « 🔑 Secrets » → ajouter GITHUB_TOKEN "        "→ activer « Accès au notebook »."    )# L'URL contient le jeton : elle ne doit JAMAIS être affichée, ni figurer dans# un message d'erreur. Les sorties de git sont donc capturées, jamais relayées.url = f"https://{jeton}@github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"if os.path.isdir(DOSSIER_PROJET):    commande = ["git", "-C", DOSSIER_PROJET, "pull", "--quiet"]else:    commande = ["git", "clone", "--quiet", url, DOSSIER_PROJET]resultat = subprocess.run(commande, capture_output=True, text=True)if resultat.returncode != 0:    raise RuntimeError(        "Récupération du code impossible.\n"        "Vérifiez que le jeton est valide et qu'il donne accès en lecture "        f"au dépôt {DEPOT_COMPTE}/{DEPOT_NOM}."    )if DOSSIER_PROJET not in sys.path:    sys.path.insert(0, DOSSIER_PROJET)print("Code récupéré :", DOSSIER_PROJET)

In [ ]:
# --- Option B : le dossier theatre_editor/ est sur votre Drive ---------# Décommentez ces lignes et ajustez le chemin, puis n'exécutez PAS l'option A.# import sys# DOSSIER_PROJET = "/content/drive/MyDrive/texte_troupe_theatre"# if DOSSIER_PROJET not in sys.path:#     sys.path.insert(0, DOSSIER_PROJET)

## 3. Configuration`config.py` porte toutes les valeurs par défaut. Les surcharges ci-dessous nevalent que pour cette session : elles ne modifient pas le fichier.**Vérifiez le dossier de travail** avant de continuer.

In [ ]:
from pathlib import Pathfrom theatre_editor import config# Dossier Drive contenant les PDF et recevant toutes les sorties.config.DOSSIER_DRIVE = Path("/content/drive/MyDrive/Troupe 122 - 2026-27")print("Dossier de travail :", config.DOSSIER_DRIVE)print("Existe             :", config.DOSSIER_DRIVE.is_dir())

## 4. Réglages typographiquesModifiez librement : cette étape est gratuite et reproductible.

In [ ]:
config.POLICE_TEXTE = "EB Garamond"config.TAILLE_TEXTE_PT = 11config.TAILLE_TITRE_ACTE_PT = 16config.TAILLE_TITRE_SCENE_PT = 14config.MARGE_CM = 3.0config.SAUT_DE_PAGE_AVANT_ACTE = Trueconfig.SAUT_DE_PAGE_AVANT_SCENE = Falsefor cle, definition in config.DEFINITIONS_STYLES.items():    saut = " + saut de page" if definition["saut_de_page"] else ""    graisse = "gras" if definition["gras"] else ("italique" if definition["italique"] else "romain")    print(f"   {definition['nom']:<14} {definition['taille_pt']:>2} pt  "          f"{definition['alignement']:<9} {graisse}{saut}")

## 5. Table d'inspection de la structure**À lire avant de générer.** Elle montre comment chaque nom en gras a étéclassé — acte, scène, personnage — et signale d'un `⚠` les classementsincertains.C'est ici qu'on repère un acte pris pour un personnage, plutôt que de ledécouvrir à la première page blanche parasite.

In [ ]:
from theatre_editor.utils import blocks, iofor chemin in io.lister_fichiers_edit(config.DOSSIER_DRIVE):    nom = io.nom_livre_depuis_edit(chemin)    index = blocks.construire_index_structure(io.lire_texte(chemin))    print("=" * 72)    print(nom)    print("=" * 72)    print(blocks.rapport_classification(index))    print()

## 6. Corriger un classementSi la table ci-dessus se trompe, forcez le classement ici. Écrivez les noms**en capitales et sans accents**, tels qu'ils apparaissent dans la colonne`LABEL`.

In [ ]:
# Exemples — décommentez et adaptez :# config.PERSONNAGES_FORCES = frozenset({"LA VOIX", "LE CHOEUR"})# config.TITRES_ACTE_FORCES = frozenset({"OUVERTURE"})# config.TITRES_SCENE_FORCES = frozenset({"ENTRACTE"})print("Personnages forcés :", sorted(config.PERSONNAGES_FORCES))print("Actes forcés       :", sorted(config.TITRES_ACTE_FORCES))print("Scènes forcées     :", sorted(config.TITRES_SCENE_FORCES))

## 7. Génération

In [ ]:
from theatre_editor import docx_exportresultats = docx_export.executer(config.DOSSIER_DRIVE)

## 8. Contrôle du documentRelit le DOCX produit et affiche le style appliqué à chaque paragraphe. Lemeilleur moyen de vérifier qu'actes, scènes et personnages sont bien distingués.

In [ ]:
import docxfor resultat in resultats:    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)    if not chemins.docx.exists():        print(f"{resultat.nom} : aucun document produit.")        continue    document = docx.Document(str(chemins.docx))    print("=" * 72)    print(f"{resultat.nom} — {len(document.paragraphs)} paragraphes")    print("=" * 72)    for paragraphe in document.paragraphs[:40]:        style = paragraphe.style.name.replace(config.PREFIXE_STYLE, "")        saut = "  [PAGE NEUVE]" if paragraphe.style.paragraph_format.page_break_before else ""        print(f"  {style:<14} | {paragraphe.text[:60]}{saut}")    if len(document.paragraphs) > 40:        print(f"  … {len(document.paragraphs) - 40} paragraphes de plus")

## 9. TéléchargementLe document est déjà sur votre Drive. Cette cellule permet de le récupérerdirectement sur votre machine.

In [ ]:
from google.colab import filesfor resultat in resultats:    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)    if chemins.docx.exists():        files.download(str(chemins.docx))

## 10. À propos de la police`python-docx` inscrit le **nom** de la police dans le document, il nel'incorpore pas. EB Garamond n'a donc pas à être installée dans Colab pour quela génération réussisse.En revanche, si elle est absente de la machine qui **ouvre** le fichier, Wordsubstituera une autre police. Pour un rendu conforme, installez EB Garamond survotre poste — elle est gratuite et disponible sur Google Fonts.

## 11. Journal

In [ ]:
# Journal détaillé de l'étape : un enregistrement par appel API, avec sa# date, son modèle, son response_id, sa durée et sa consommation de jetons.import jsonchemin = config.DOSSIER_DRIVE / config.NOM_JOURNAL.format(etape="docx")if chemin.exists():    journal = json.loads(chemin.read_text(encoding="utf-8"))    print("Dernière exécution :", journal["derniere_execution"])    print("Configuration      :", json.dumps(journal["configuration"], ensure_ascii=False))    print()    for nom, bilan in journal["livres"].items():        print(f"{nom} : {json.dumps(bilan, ensure_ascii=False)}")    jetons = sum(        (appel.get("tokens_entree") or 0) + (appel.get("tokens_sortie") or 0)        for appel in journal["appels"]    )    print()    print(f"{len(journal['appels'])} appel(s) journalisé(s), {jetons:,} jetons".replace(",", " "))else:    print("Aucun journal : l'étape n'a pas encore été lancée.")